In [ ]:
!pip install -q chromadb openai pandas

In [ ]:
import pandas as pd

from chromadb import PersistentClient
from pprint import pprint
from pydantic import BaseModel
from rich.console import Console
from rich.table import Table

## Load the dataset

We are using the [Goodreads Top 100 Classics](https://www.kaggle.com/datasets/notkrishna/goodreads-top-100-classical-books-of-all-time) dataset from Kaggle.

In [ ]:
class Book(BaseModel):
    title: str
    author: str
    description: str

In [ ]:
PATH_TO_DATASET = "/content/goodreads_dataset.csv"

df = pd.read_csv(PATH_TO_DATASET)

# Filter books without description
df = df[df['description'].notna()]

In [ ]:
books = [Book(**row) for row in df.to_dict(orient="records")]

In [ ]:
pprint(books)

In [ ]:
max_description_length = max(len(b.description) for b in books)
print(f"Max description length: {max_description_length}")

## Create ChromaDB collection

In [ ]:
PATH_TO_CHROMA_DB = "/content/chromadb"
chroma_client = PersistentClient(path=PATH_TO_CHROMA_DB)

In [ ]:
chroma_collection = chroma_client.get_or_create_collection(name="goodreads-experiments")

## Load books into the collection

In [ ]:
# Identify books by one-based index
ids = [f"{i + 1}" for i in range(len(books))]
metadatas = [{ "title": b.title, "author": b.author } for b in books]
documents = [b.description for b in books]

chroma_collection.upsert(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

In [ ]:
results = chroma_collection.get(include=["documents", "metadatas", "embeddings"])

pd.DataFrame({
    "id": results["ids"],
    "document": results["documents"],
    "metadata": results["metadatas"],
    "embeddings": results["embeddings"].tolist()
})

## Query books

In [ ]:
def print_query_results(result):
    console = Console()

    queries_count = len(result["ids"])
    for i in range(queries_count):
        table = Table(show_lines=True, expand=True)
        table.add_column("#")
        table.add_column("Title")
        table.add_column("Author")
        table.add_column("Description")

        results_count = len(result["ids"][i])
        for j in range(results_count):
            table.add_row(result["ids"][i][j], result["metadatas"][i][j]["title"], result["metadatas"][i][j]["author"], result["documents"][i][j])

        console.print(table)

In [ ]:
query_result = chroma_collection.query(
    query_texts=[
        "Novels about war seen through ordinary people",
        "Find books with magical realism or surreal elements"
    ],
    n_results=10,
    include=["metadatas", "documents"]
)

In [ ]:
print_query_results(query_result)